# Notebook 14 — Variance Preservation

## The Core Argument

MICE imputes **conditional means**: E[X_j | X_{-j}]. This is optimal for RMSE
but systematically suppresses marginal variance:

$$\text{Var}(\hat{X}_j^{\text{MICE}}) < \text{Var}(X_j^{\text{true}})$$

Van Buuren (2018, §2.6) states this explicitly:
> "The minimum mean squared error is achieved by regression imputation, which
> suppresses variance and produces biased statistical inference."

MIGA imputes from the **empirical bootstrap distribution** of observed values.
The expected variance of the imputed values equals the variance of the observed
marginal distribution — which approximates the true variance.

**Measurement:** Variance ratio = Var(imputed) / Var(true). Closer to 1.0 is better.
MICE < 1.0 (variance suppression). MIGA ≈ 1.0 (variance preservation).

**Why this matters:** Any downstream analysis that uses variance of the imputed
data — confidence intervals, hypothesis tests, distributional comparisons — will
be biased if MICE is used. MIGA avoids this bias.

**Run script first:**
```
.venv/bin/python scripts/run_variance_preservation.py
```
Saves `results/14_variance_preservation.json`. Takes ~20 min.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load Results

In [ ]:
VAR_PATH = os.path.join(RESULTS_DIR, "14_variance_preservation.json")
if not os.path.exists(VAR_PATH):
    print("Run scripts/run_variance_preservation.py first.")
    var_data = {}
else:
    with open(VAR_PATH) as f:
        var_data = json.load(f)
    print(f"Loaded: {list(var_data.keys())}")

## 2. Variance Ratio Table — Per Dataset and Method

In [ ]:
METHODS = ["Mean", "KNN", "MICE", "MIGA"]

if var_data:
    print("Variance Ratio = Var(imputed) / Var(true)  [1.0 = perfect, <1 = suppressed]")
    print()
    print(f"  {'Dataset':<12}  {'Feature(s)':<30}  {'Mean':>7}  {'KNN':>7}  {'MICE':>7}  {'MIGA':>7}")
    print("  " + "-" * 75)
    for ds, data in var_data.items():
        miss_cols = data.get("miss_cols", [])
        feat_str  = ", ".join(miss_cols[:3]) + ("…" if len(miss_cols) > 3 else "")
        vals = {m: data.get(m, {}).get("mean_ratio", float("nan")) for m in METHODS}
        best = min(METHODS, key=lambda m: abs(vals[m] - 1.0) if vals[m]==vals[m] else float("inf"))
        row = "  ".join(f"{vals[m]:>7.4f}{'★' if m==best else ' '}" for m in METHODS)
        print(f"  {ds:<12}  {feat_str:<30}  {row}")
    print()
    print("  ★ = closest to 1.0 (best variance preservation)")

## 3. Deviation from True Variance

In [ ]:
if var_data:
    print("|ratio - 1.0|  (lower = better)")
    print()
    print(f"  {'Dataset':<12}  {'Mean':>8}  {'KNN':>8}  {'MICE':>8}  {'MIGA':>8}  {'MIGA wins?'}")
    print("  " + "-" * 60)
    miga_wins = 0
    for ds, data in var_data.items():
        devs = {m: abs(data.get(m, {}).get("mean_ratio", float("nan")) - 1.0)
                for m in METHODS}
        best = min(devs, key=lambda m: devs[m] if devs[m]==devs[m] else float("inf"))
        win = "✓" if best == "MIGA" else f"({best})"
        if best == "MIGA":
            miga_wins += 1
        print(f"  {ds:<12}  {devs['Mean']:>8.4f}  {devs['KNN']:>8.4f}  "
              f"{devs['MICE']:>8.4f}  {devs['MIGA']:>8.4f}  {win}")
    print(f"\n  MIGA wins variance preservation on {miga_wins}/{len(var_data)} datasets")

## 4. Per-Feature Variance Ratios

In [ ]:
if var_data:
    for ds, data in var_data.items():
        miss_cols = data.get("miss_cols", [])
        if not miss_cols:
            continue
        print(f"\n{ds}:")
        print(f"  {'Feature':<15}  {'True Var':>10}  {'Mean':>8}  {'KNN':>8}  {'MICE':>8}  {'MIGA':>8}")
        print("  " + "-" * 60)
        for col in miss_cols:
            true_v = data.get("true_var", {}).get(col, float("nan"))
            vals = {m: data.get(m, {}).get("ratios", {}).get(col, float("nan"))
                    for m in METHODS}
            print(f"  {col:<15}  {true_v:>10.3f}  "
                  + "  ".join(f"{vals[m]:>8.4f}" for m in METHODS))

## 5. Visualisation — Variance Ratio Comparison

In [ ]:
if var_data:
    n_ds = len(var_data)
    fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
    if n_ds == 1:
        axes = [axes]

    colours = {"Mean": "tab:gray", "KNN": "tab:green", "MICE": "tab:orange", "MIGA": "tab:blue"}

    for ax, (ds, data) in zip(axes, var_data.items()):
        miss_cols = data.get("miss_cols", [])
        x = np.arange(len(miss_cols))
        width = 0.18

        for i, method in enumerate(METHODS):
            ratios = [data.get(method, {}).get("ratios", {}).get(c, float("nan"))
                      for c in miss_cols]
            ax.bar(x + (i - 1.5) * width, ratios, width,
                   label=method, color=colours[method], alpha=0.85)

        ax.axhline(1.0, color="black", lw=2, ls="--", label="True = 1.0")
        ax.set_xticks(x)
        ax.set_xticklabels(miss_cols, rotation=30, ha="right", fontsize=9)
        ax.set_title(ds)
        ax.set_ylabel("Var(imputed) / Var(true)")
        ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    plt.suptitle("Variance Ratio by Method and Feature  (1.0 = perfect preservation)",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "14_variance_ratios.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/14_variance_ratios.png")

## 6. Interpretation

**MICE variance suppression:** MICE imputes E[X_j | X_{-j}]. Since conditional
expectations have lower variance than the marginal: Var(E[X|Y]) ≤ Var(X) (law of
total variance). For highly correlated features (Iris: r ≈ 0.8), MICE can suppress
variance by 30–50%.

**MIGA variance preservation:** MIGA samples from the bootstrap distribution of
observed values. The bootstrap distribution approximates the true marginal distribution,
so variance is approximately preserved. The bootstrap naturally captures discreteness
(Haberman Nodes: 44% zeros) and multi-modality (Iris petal length: trimodal).

**Practical implication:** Use MICE when you need minimum RMSE on individual imputed
values. Use MIGA when you need the imputed dataset to have the correct marginal
distributions — e.g. for:
- Computing confidence intervals from multiply-imputed data
- Testing distributional hypotheses (KS test, etc.)
- Generating plausible synthetic samples from the imputed distribution
- Any analysis that uses variance of the imputed values